In [ ]:
import pandas as pd
import csv
import statistics
import matplotlib.pyplot as plt
from matplotlib.axes import Axes
import os
import numpy as np
from pathlib import Path
from xmigcs import XMIGCS_ROOT_DIR

# 复制日志文件到本地
# scp ubuntu@10.11.218.152:/home/ubuntu/XMIGCS_DEPLOY/xmigcs/logs/navigate/20260422_155648_855669_commands.csv .

log_dir = f"{XMIGCS_ROOT_DIR}/logs/navigate"
csv_path = f"{log_dir}/20260422_092755_101299_commands.csv"
csv_path = Path(csv_path)

save_fig = False
save_format = "svg"

df = pd.read_csv(csv_path, header=None)
# print(df)

In [ ]:
"fps analyze"
fps_list = []
with open(csv_path, "r") as f:
    reader = csv.reader(f)
    for row in reader:
        fps = float(row[1])
        fps_list.append(fps)

avg_fps = statistics.mean(fps_list)
std_fps = statistics.stdev(fps_list)

print(f"mean FPS: {avg_fps:.2f}")
print(f"std: {std_fps:.2f}")


# 过滤主峰
filtered_fps_list = [fps for fps in fps_list if not (95 <= fps <= 100)]
# 创建左右两个子图
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1: Axes
ax2: Axes

# 左侧：原始数据的百分比直方图
weights_all = np.ones(len(fps_list)) / len(fps_list) * 100
ax1.hist(fps_list, bins=50, weights=weights_all, edgecolor="black")
ax1.set_xlabel("fps")
ax1.set_ylabel("percent (%)")
ax1.set_title("All data")

# 右侧：过滤主峰后的百分比直方图（分母仍为总样本数）
weights_filtered = np.ones(len(filtered_fps_list)) / len(fps_list) * 100
n, bins = np.histogram(filtered_fps_list, bins=50, weights=weights_filtered)
ax2.hist(fps_list, bins=50, weights=weights_all, edgecolor="black")
ax2.set_ylim([0, max(n)])
ax2.set_xlabel("fps")
ax2.set_ylabel("percent (%)")
ax2.set_title("Ignoring 95~100")

# 可选：设置相同的x轴范围，便于比较
x_min = min(min(fps_list), min(filtered_fps_list)) if filtered_fps_list else min(fps_list)
x_max = max(max(fps_list), max(filtered_fps_list)) if filtered_fps_list else max(fps_list)
ax1.set_xlim(x_min, x_max)
ax2.set_xlim(x_min, x_max)
ax1.grid(linestyle="--")
ax2.grid(linestyle="--")

plt.tight_layout()
plt.show()
if save_fig:
    save_dir = os.path.join(log_dir, csv_path.name.split(".")[0])
    os.makedirs(save_dir, exist_ok=True)
    file_name_no_ext = csv_path.name.split(".")[0]
    fig_name = f"fps_analyze.{save_format}"
    fig_path = f"{save_dir}/{fig_name}"
    fig.savefig(fig_path)
    print(f"saved: {fig_path}")
plt.close()

In [ ]:
"过滤没有走路的数据"
# 提取第8列为True的行号
true_rows = df[df[8] == True].index.tolist()
print(f"True总行数: {len(true_rows)}")
print(f"True行号示例: {true_rows[:10]}")

# 找出连续True的起始和结束点
segments = []
if len(true_rows) > 0:
    start = true_rows[0]
    end = true_rows[0]
    for i in range(1, len(true_rows)):
        if true_rows[i] == end + 1:  # 连续
            end = true_rows[i]
        else:  # 断开
            segments.append([start, end])
            start = true_rows[i]
            end = true_rows[i]
    segments.append([start, end])  # 最后一个段

print(f"连续True时间段: {segments}")
print(f"共{len(segments)}个连续时间段")

# TODO: 去除后四个数据全为0的片段
# 去除后四个数据全为0的片段
filtered_segments = []
for start, end in segments:
    segment_data = df.iloc[start:end + 1]
    last_4_cols = segment_data.iloc[:, -4:].values
    # 检查最后4列是否全为0
    if not (last_4_cols == 0).all():
        filtered_segments.append([start, end])

print(f"过滤后时间段: {filtered_segments}")
print(f"共{len(filtered_segments)}个时间段")
segments = filtered_segments

In [ ]:
def plot_segment(df, segments, seg_idx):
    """绘制指定时间段的数据

    Args:
        df: DataFrame
        segments: 连续True时间段列表
        seg_idx: 时间段索引(从0开始)
    """
    import matplotlib.pyplot as plt

    start, end = segments[seg_idx]
    segment_data = df.iloc[start:end + 1].copy()

    time_col = segment_data[0].values
    last_4_cols = segment_data.iloc[:, -4:].values

    relative_time = time_col - time_col[0]
    fig, ax = plt.subplots(figsize=(14, 6))
    labels = ["walk_vel_ref", "pos_x", "pos_y", "heading"]
    for i in range(4):
        ax.plot(relative_time, last_4_cols[:, i], label=labels[i])

    ax.set_xlabel('Relative Time (s)')
    ax.set_ylabel('Value')
    length = end - start + 1
    ax.set_title(f'Segment {seg_idx}: [{start}, {end}], start_time={time_col[0]:.3f}, length={length}')

    ax.legend()
    ax.grid(True)
    plt.tight_layout()

    plt.show()
    plt.close()

    if save_fig:
        # 保存图片
        save_path = os.path.join(save_dir, f'segment_{seg_idx}.{save_format}')
        fig.savefig(save_path)
        print(f"图片已保存到: {save_path}")

# 用for循环绘制所有时间段
for i in range(len(segments)):
    plot_segment(df, segments, i)
# plot_segment(df, segments, 0)